In [0]:
import pyspark.sql.functions as F
import pyspark.sql.types as T
from datetime import datetime

#1. Load bronze.stop_times data


In [0]:
df = spark.table('warsaw_gtfs.bronze.stop_times')
df.display()

In [0]:
dfduplicates = df.groupBy(df.columns).count().filter('count > 1')
dfduplicates.display()

In [0]:
#data issues
# 1.Arrival and departure times need to be validated
# 2.Unfriendly pickup and dropoff values.

#2. Transformations

## Validate arrival and departure times

In [0]:
# define udf
def check_arrival(arrival_time, departure_time):
    arrival_time_parts = arrival_time.split(':')
    departure_time_parts = departure_time.split(':')
    for j in range(len(arrival_time_parts)):
        if arrival_time_parts[j] <= departure_time_parts[j]:
            return True
        else:
            continue
    return False

# register udf
check_arrival_udf = F.udf(check_arrival, T.BooleanType())

# validate departure times, should be after arrival
df.filter(check_arrival_udf(F.col('arrival_time'), F.col('departure_time')) == False).display()

## Map more friendly values to pickup and dropoff values

In [0]:
df2 = df.withColumn(
    'pickup_type',
    F.when(F.col("pickup_type") == 0, 'regular')
    .when(F.col('pickup_type') == 1, 'no boarding/alighting')
    .when(F.col('pickup_type') == 3, 'on demand')
    .otherwise('n/a')
).withColumn(
    'drop_off_type',
    F.when(F.col("drop_off_type") == 0, 'regular')
    .when(F.col('drop_off_type') == 1, 'no boarding/alighting')
    .when(F.col('drop_off_type') == 3, 'on demand')
    .otherwise('n/a')
)
df2.display()



In [0]:
# check if any pickup_type differs from drop_off_type for a row, if not drop one
df2.filter(F.col('pickup_type') != F.col('drop_off_type')).display()

df3 = df2.drop('drop_off_type')
df3.display()


In [0]:
# validate data, no trip_id should have more that one of the same stop sequence

df3.groupBy('trip_id', 'stop_sequence').count().filter('count > 1').display()

#3. Load data to silver layer

In [0]:
df3.write.mode('overwrite').saveAsTable('warsaw_gtfs.silver.stop_times')